In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC

from sklearn.metrics import (
    roc_auc_score,
    roc_curve,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
    classification_report,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay
)

from imblearn.over_sampling import SMOTE
from xgboost import XGBClassifier


In [ ]:
df = pd.read_csv("creditcard.csv")

print("Dataset shape:", df.shape)
display(df.head())


In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum().sum())

print("\nClass distribution:")
print(df["Class"].value_counts())

print("\nClass percentage:")
print((df["Class"].value_counts(normalize=True) * 100).round(4))


In [ ]:
X = df.drop("Class", axis=1)
y = df["Class"]

print("Features shape:", X.shape)
print("Target shape:", y.shape)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", len(X_train))
print("Testing samples :", len(X_test))


In [ ]:
# Scale Time and Amount
scaler = StandardScaler()

X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

X_train_scaled[["Time", "Amount"]] = scaler.fit_transform(
    X_train[["Time", "Amount"]]
)

X_test_scaled[["Time", "Amount"]] = scaler.transform(
    X_test[["Time", "Amount"]]
)


In [ ]:
# Apply SMOTE only to the training data
smote = SMOTE(random_state=42)

X_train_smote, y_train_smote = smote.fit_resample(
    X_train_scaled,
    y_train
)

print("Original training distribution:")
print(y_train.value_counts())

print("\nAfter SMOTE:")
print(pd.Series(y_train_smote).value_counts())


In [ ]:
xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_smote, y_train_smote)

print("XGBoost training completed.")


In [ ]:
y_prob_xgb = xgb_model.predict_proba(X_test_scaled)[:, 1]
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)

roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)
pr_auc_xgb = average_precision_score(y_test, y_prob_xgb)

print(f"XGBoost ROC-AUC: {roc_auc_xgb:.4f}")
print(f"XGBoost PR-AUC : {pr_auc_xgb:.4f}")


In [ ]:
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("XGBoost Confusion Matrix:")
print(cm_xgb)

print("\nClassification Report:")
print(classification_report(y_test, y_pred_xgb, zero_division=0))


In [ ]:
ConfusionMatrixDisplay(
    confusion_matrix=cm_xgb,
    display_labels=["Genuine", "Fraud"]
).plot()

plt.title("XGBoost Confusion Matrix")
plt.show()


In [ ]:
fpr_xgb, tpr_xgb, _ = roc_curve(y_test, y_prob_xgb)

plt.figure(figsize=(7, 5))
plt.plot(
    fpr_xgb,
    tpr_xgb,
    label=f"XGBoost (AUC = {roc_auc_xgb:.3f})"
)
plt.plot([0, 1], [0, 1], linestyle="--", label="Random Classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve - Credit Card Fraud Detection")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
precision_xgb, recall_xgb, _ = precision_recall_curve(
    y_test,
    y_prob_xgb
)

plt.figure(figsize=(7, 5))
plt.plot(
    recall_xgb,
    precision_xgb,
    label=f"XGBoost (PR-AUC = {pr_auc_xgb:.3f})"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve - Credit Card Fraud Detection")
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
# SVM baseline
svm_model = Pipeline([
    ("scaler", StandardScaler()),
    ("classifier", SVC(
        kernel="rbf",
        probability=True,
        class_weight="balanced",
        random_state=42
    ))
])

svm_model.fit(X_train, y_train)

y_prob_svm = svm_model.predict_proba(X_test)[:, 1]
y_pred_svm = (y_prob_svm >= 0.5).astype(int)

roc_auc_svm = roc_auc_score(y_test, y_prob_svm)
pr_auc_svm = average_precision_score(y_test, y_prob_svm)

print(f"SVM ROC-AUC: {roc_auc_svm:.4f}")
print(f"SVM PR-AUC : {pr_auc_svm:.4f}")


In [ ]:
comparison = pd.DataFrame({
    "Model": ["XGBoost", "SVM"],
    "ROC-AUC": [roc_auc_xgb, roc_auc_svm],
    "PR-AUC": [pr_auc_xgb, pr_auc_svm]
})

display(comparison)


In [ ]:
for threshold in [0.10, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    y_threshold = (y_prob_xgb >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_threshold
    ).ravel()

    precision = precision_score(
        y_test,
        y_threshold,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        y_threshold,
        zero_division=0
    )

    print(
        f"Threshold={threshold:.2f} | "
        f"TP={tp}, FP={fp}, FN={fn}, TN={tn} | "
        f"Precision={precision:.3f}, Recall={recall:.3f}"
    )


In [ ]:
# XGBoost feature importance
importance = pd.Series(
    xgb_model.feature_importances_,
    index=X.columns
).sort_values(ascending=False)

print("Top 15 features:")
display(importance.head(15))

plt.figure(figsize=(8, 6))
importance.head(15).sort_values().plot(kind="barh")
plt.xlabel("Importance")
plt.title("Top 15 XGBoost Feature Importances")
plt.show()
